# 04 — Hybrid & CSQE Figures (Ch 4.5, 4.6, 4.7)

**Updated 2026-05-31:**
- Config A/B/C labels renamed to **BM25-only-expanded / Dense-only-expanded / Both-expanded** (Decision D3, Task 2.4).
- Numeric rounding to 3 decimals in tables (Decision D4): mDPR 0.499, R@100 0.947.

**Outputs:**
- Table 4.4 (LaTeX) — hybrid baselines
- Fig 4.9 v1/v2 — CC α sweep
- Table 4.5 (LaTeX) — CSQE ablation
- Fig 4.10 v1 — CSQE α sweep (recommended cut from final thesis)
- Table 4.7 (LaTeX) — Configs A/B/C → descriptive names
- Fig 4.11 v1/v2/v3 — system progression (headline)

In [ ]:
import sys, re
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from _helpers import *

# Decision D3 — Config A/B/C → descriptive names.
# Applied to raw CSVs at render time; source files keep Config A/B/C labels.
CONFIG_RENAME = {
    'A: BM25+CSQE + Dense': 'BM25-only-expanded',
    'B: BM25 + Dense+CSQE': 'Dense-only-expanded',
    'C: BM25+CSQE + Dense+CSQE': 'Both-expanded',
}
def relabel_config(s: str) -> str:
    for old, new in CONFIG_RENAME.items():
        if old in s:
            s = s.replace(old, new)
    return s

## Table 4.4 — Hybrid baselines (no QE)

In [ ]:
hyb_summary = pd.read_csv(DATA_RAW / 'exp12_summary.csv')
print(hyb_summary)
(OUTPUT_PDF / 'table_4_4.tex').write_text(
    hyb_summary.to_latex(index=False, float_format='%.3f', column_format='lcccc'),
    encoding='utf-8')
print('saved: table_4_4.tex')

## Fig 4.9 — Hybrid CC α sweep

In [ ]:
cc = pd.read_csv(DATA_RAW / 'exp12_cc_sweep.csv')
# First column is alpha
alpha_col = cc.columns[0]
best = cc.loc[cc['nDCG@10'].idxmax()]
best_alpha = best[alpha_col]

# v1 — NDCG only
fig, ax = plt.subplots()
ax.plot(cc[alpha_col], cc['nDCG@10'], marker='o', color='#1f1f1f')
ax.axvline(best_alpha, color='#8c8c8c', linestyle='--', linewidth=1,
           label=f'Best \u03b1={best_alpha:.1f}, NDCG@10={best["nDCG@10"]:.3f}')
ax.set_xlabel('\u03b1 (mDPR weight)')
ax.set_ylabel('NDCG@10')
ax.legend(loc='lower center')
save_fig(fig, 'fig_4_9_alpha_sweep_v1')

# v2 — all four metrics
fig, ax = plt.subplots()
metric_cols = ['nDCG@10', 'Recall@10', 'Recall@100', 'MRR']
styles = [dict(color='#1f1f1f', marker='o'),
          dict(color='#4d4d4d', marker='s', linestyle='--'),
          dict(color='#8c8c8c', marker='^', linestyle=':'),
          dict(color='#b3b3b3', marker='d', linestyle='-.')]
for col, st in zip(metric_cols, styles):
    if col in cc.columns:
        ax.plot(cc[alpha_col], cc[col], label=col, **st)
ax.set_xlabel('\u03b1 (mDPR weight)')
ax.set_ylabel('Score')
ax.legend(loc='lower center', ncol=2)
save_fig(fig, 'fig_4_9_alpha_sweep_v2_all')

## Table 4.5 — CSQE ablation

In [ ]:
ablation = pd.read_csv(DATA_RAW / 'csqe_ablation_table.csv')
# Apply Config rename in case any system labels mention Config A
ablation['System'] = ablation['System'].apply(relabel_config)
print(ablation)
(OUTPUT_PDF / 'table_4_5.tex').write_text(
    ablation.to_latex(index=False, float_format='%.3f', column_format='lcccc'),
    encoding='utf-8')
print('saved: table_4_5.tex')

## Fig 4.10 — CSQE α sweep
**Recommendation:** cut from final thesis. The curve is so flat (Δ≈0.002 across α=1..4) it should be one sentence in §3.8. We render v1 anyway for the registry.

In [ ]:
alpha = pd.read_csv(DATA_RAW / 'csqe_alpha_ablation.csv')
fig, ax = plt.subplots()
ax.plot(alpha['alpha'], alpha.iloc[:, 1], marker='o', label='BM25+CSQE', color='#8c8c8c')
ax.plot(alpha['alpha'], alpha.iloc[:, 2], marker='s', label='Best system (RRF k=20)', color='#1f1f1f')
ax.set_xlabel('\u03b1 (CSQE query repetition)')
ax.set_ylabel('NDCG@10')
ax.set_xticks([1, 2, 3, 4])
ax.legend(loc='center right')
save_fig(fig, 'fig_4_10_csqe_alpha_v1')

## Table 4.7 — Configs A/B/C → descriptive names

In [ ]:
summary = pd.read_csv(DATA_RAW / 'exp21_summary.csv')
summary['Method'] = summary['Method'].apply(relabel_config)
print(summary.to_string(index=False))
(OUTPUT_PDF / 'table_4_7.tex').write_text(
    summary.to_latex(index=False, float_format='%.3f', column_format='lcccc'),
    encoding='utf-8')
print('saved: table_4_7.tex')

## Fig 4.11 — System progression (headline)

In [ ]:
story_rows = [
    ('BM25', 0.462),
    ('mDPR', 0.499),
    ('Best blind Dense\n(Aya 8B Q2D)', 0.616),
    ('Hybrid RRF\n(no QE)', 0.627),
    ('BM25+CSQE', 0.616),
    ('Best system\n(BM25-only-expanded RRF)', 0.714),
]
labels = [r[0] for r in story_rows]
vals = [r[1] for r in story_rows]
colors = ['#b3b3b3'] * 5 + ['#1f1f1f']

# v1 — plain bar
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, vals, color=colors, edgecolor='black')
ax.set_ylabel('NDCG@10')
ax.set_ylim(0, 0.8)
plt.xticks(rotation=15, ha='right')
save_fig(fig, 'fig_4_11_progression_v1')

# v2 — with Δ annotations to the previous best-so-far
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, vals, color=colors, edgecolor='black')
best_so_far = -np.inf
for i, v in enumerate(vals):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    if i > 0:
        d = v - best_so_far
        if d > 0:
            ax.text(i, 0.02, f'+{d:.3f}', ha='center', va='bottom', fontsize=7, color='#4d4d4d')
    best_so_far = max(best_so_far, v)
ax.set_ylabel('NDCG@10')
ax.set_ylim(0, 0.85)
plt.xticks(rotation=15, ha='right')
save_fig(fig, 'fig_4_11_progression_v2_annot')

# v3 — grouped multi-metric
row_filter = ['BM25 alone', 'mDPR alone', 'Hybrid RRF k=20 (Exp 1.2)',
              'BM25+CSQE (Exp 013)',
              'BM25-only-expanded RRF (k=20)']  # post-rename
sub = summary[summary.Method.isin(row_filter)].copy()
sub['short'] = ['BM25', 'mDPR', 'Hybrid', 'BM25+CSQE', 'Best system']
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(sub))
w = 0.2
metric_pairs = [('nDCG@10', '#1f1f1f'), ('Recall@10', '#4d4d4d'),
                ('Recall@100', '#8c8c8c'), ('MRR', '#b3b3b3')]
for i, (col, c) in enumerate(metric_pairs):
    ax.bar(x + (i - 1.5) * w, sub[col], w, label=col, color=c, edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(sub.short)
ax.set_ylabel('Score')
ax.legend(loc='lower right', ncol=2, fontsize=8)
ax.set_ylim(0, 1.05)
save_fig(fig, 'fig_4_11_progression_v3_grouped')